# Multistart NUTS on the UCI BNN — one basin or many?

5 independent NUTS chains, each started from its **own draw from the prior** (no MAP reference), from `sazz/gpu_friendly/scripts/mcmc_test.py`.

**Question:** do the chains converge to the same posterior, or do they settle in different basins?

### Why raw weight-space comparison is the wrong tool
The BNN posterior is invariant under hidden-unit permutations and (with `tanh`) sign flips. Two weight vectors that are permutation/sign images of each other encode the **identical function** and sit in the **same basin**. So:
- Raw weight marginals disagreeing across chains proves nothing — it is mostly relabelling.
- Raw-weight R-hat will be huge by construction. Expected. Not a convergence failure.

### What actually answers the question
1. **Functional space** — push draws through the net; compare per-chain predictive. Same function ⇒ same basin (or functionally-equivalent basins). Different function ⇒ genuinely different mode.
2. **Permutation-invariant weight summaries** — `log_sigma`, `‖θ‖`, singular-value spectra of each weight matrix, sorted per-unit weight norms. A difference in these *is* a real difference.
3. **R-hat / ESS on the predictions** (not the weights) — the clean quantitative "same posterior?" number.
4. **Within-chain traces** of functional scalars — did a single chain visit more than one basin? (NUTS rarely does for BNNs.)

Triangle plots appear in section 6, built **only on permutation-invariant scalars**, so their lobes mean something.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from Filippo_plotting.mcmc_plots import better_pairs

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "font.size": 10,
})

RESULTS_DIR = Path("results/mcmc_test")
CHAIN_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#17becf", "#8c564b"]

# mcmc_test.py writes to results/mcmc_test/<dataset>/<variant>/nuts_chains.pt
# (variant is 'small' | 'medium' | 'deep_narrow' | 'deep_wide'). Grab every one.
run_files = sorted(RESULTS_DIR.glob("*/*/nuts_chains.pt"))
print(f"found {len(run_files)} run file(s):")
for p in run_files:
    print("  ", p)
if not run_files:
    print("\n  -> run:  python -m sazz.gpu_friendly.scripts.mcmc_test --collect-warmup")

In [ ]:
# Which run(s) to analyse. Default: all found. Set e.g. SELECT = ["boston/medium"] to focus.
SELECT = None


def load_run(path: Path) -> dict:
    d = torch.load(path, weights_only=False)
    return {
        "tag": f"{d['dataset']}/{path.parent.name}",
        "dataset": d["dataset"],
        "variant": path.parent.name,
        "layer_sizes": list(d["layer_sizes"]),
        "activation": d["activation"],
        "y_std": float(d["y_std"]),
        "split_id": int(d.get("split_id", 0)),
        "n_chains": int(d["n_chains"]),
        "n_draws": int(d["n_draws"]),
        "n_warmup": int(d["n_warmup"]),
        "chain_seeds": list(d["chain_seeds"]),
        "samples": np.asarray(d["samples"], dtype=np.float64),          # [C, N, D]
        "init_points": np.asarray(d["init_points"], dtype=np.float64),   # [C, D]
        "warmup_samples": (np.asarray(d["warmup_samples"], dtype=np.float64)
                            if d.get("warmup_samples") is not None else None),
        "diverging": np.asarray(d["diverging"], dtype=bool),
        "num_steps": np.asarray(d["num_steps"], dtype=np.int64),
        "accept_prob": np.asarray(d["accept_prob"], dtype=np.float64),
    }


runs = {}
for p in run_files:
    r = load_run(p)
    if SELECT is not None and r["tag"] not in SELECT:
        continue
    runs[r["tag"]] = r

for tag, r in runs.items():
    div = r["diverging"].sum(axis=1)
    print(f"{tag:20s} layers={r['layer_sizes']}  {r['n_chains']}x{r['n_draws']} draws  "
          f"D={r['samples'].shape[-1]}  divergences/chain={div.tolist()}  "
          f"warmup={'yes' if r['warmup_samples'] is not None else 'no'}")

In [ ]:
# ---------------------------------------------------------------------------
# Param layout + network forward pass. Flat order is [W0,b0,W1,b1,...,log_sigma]
# exactly as mcmc_test.py::_flatten_posterior writes it.
# ---------------------------------------------------------------------------

def split_params(vec: np.ndarray, layer_sizes: list[int]):
    """vec: [..., D] -> (list of W [..., n_out, n_in], list of b [..., n_out], log_sigma [...]).
    Works batched over any leading shape."""
    Ws, bs = [], []
    off = 0
    lead = vec.shape[:-1]
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        w = vec[..., off:off + n_out * n_in].reshape(*lead, n_out, n_in); off += n_out * n_in
        b = vec[..., off:off + n_out]; off += n_out
        Ws.append(w); bs.append(b)
    log_sigma = vec[..., off]
    return Ws, bs, log_sigma


def forward_batch(X: np.ndarray, Ws, bs, activation: str) -> np.ndarray:
    """X: [N_in_pts, n_features]. Ws[l]: [S, n_out, n_in], bs[l]: [S, n_out].
    Returns [S, N_in_pts] network outputs (last layer width 1, squeezed)."""
    # h: [S, N_pts, width]
    h = np.broadcast_to(X, (Ws[0].shape[0],) + X.shape).copy()
    for li, (w, b) in enumerate(zip(Ws, bs)):
        h = np.einsum("spi,soi->spo", h, w) + b[:, None, :]
        if li < len(Ws) - 1:
            h = np.tanh(h) if activation == "tanh" else np.maximum(0.0, h)
    return h[..., 0]


def chain_predictions(samples_c: np.ndarray, X: np.ndarray, layer_sizes, activation,
                       n_pred: int = 400) -> np.ndarray:
    """samples_c: [N_draws, D] for ONE chain. Returns [n_pred, len(X)] predictions
    from n_pred evenly-spaced draws."""
    sel = np.linspace(0, samples_c.shape[0] - 1, min(n_pred, samples_c.shape[0])).round().astype(int)
    Ws, bs, _ = split_params(samples_c[sel], layer_sizes)
    return forward_batch(X, Ws, bs, activation)


print("forward helpers loaded")

In [ ]:
# ---------------------------------------------------------------------------
# Rebuild the exact train split each run used, so we can push draws through
# the net on real inputs. make_split is seeded by BASE_SEED + split_id.
# ---------------------------------------------------------------------------
from sazz.gpu_friendly.scripts.uci_bnn_grid import load_raw_datasets, make_split, BASE_SEED

_raw_cache = {}


def get_split(dataset: str, split_id: int):
    key = (dataset, split_id)
    if key in _raw_cache:
        return _raw_cache[key]
    raw = load_raw_datasets((dataset,))
    if dataset not in raw:
        _raw_cache[key] = None
        return None
    X, y = raw[dataset]
    data = make_split(X, y, seed=BASE_SEED + split_id, dtype=torch.float64, device="cpu")
    out = {k: (v.cpu().numpy() if hasattr(v, "cpu") else v) for k, v in data.items()}
    _raw_cache[key] = out
    return out


for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    if sp is None:
        print(f"{tag}: raw data for '{r['dataset']}' NOT found -- functional sections will skip it")
    else:
        print(f"{tag}: X_train {sp['X_train'].shape}, X_test {sp['X_test'].shape}")

## 1. Per-chain posterior predictive band (the headline)

Each chain's draws pushed through the network on the training inputs, sorted by the first standardized feature. Per-chain predictive **mean ± 90% band**.

- **Bands overlap for all 5 chains** → same posterior over functions. Whatever weight-space disagreement exists is relabelling, not different basins.
- **One or more bands systematically offset** → that chain found a genuinely different fit (a different functional basin).

(All in standardized-`y` units; multiply by `y_std` for raw units.)

In [ ]:
N_PRED_DRAWS = 400

for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    if sp is None:
        continue
    Xtr, ytr = sp["X_train"], sp["y_train"]
    order = np.argsort(Xtr[:, 0])
    Xs, ys, xaxis = Xtr[order], ytr[order], Xtr[order][:, 0]

    fig, ax = plt.subplots(figsize=(11, 4.6))
    ax.scatter(xaxis, ys, s=9, color="0.35", alpha=0.4, label="train y", zorder=1)
    for c in range(r["n_chains"]):
        preds = chain_predictions(r["samples"][c], Xs, r["layer_sizes"], r["activation"], N_PRED_DRAWS)
        mu = preds.mean(0)
        lo, hi = np.percentile(preds, [5, 95], axis=0)
        col = CHAIN_COLORS[c % len(CHAIN_COLORS)]
        ax.plot(xaxis, mu, color=col, lw=1.5, label=f"chain {c}", zorder=3)
        ax.fill_between(xaxis, lo, hi, color=col, alpha=0.12, zorder=2)
    ax.set_title(f"{tag}: per-chain posterior predictive mean ± 90% band  (layers={r['layer_sizes']})")
    ax.set_xlabel("x[:,0] (standardized)"); ax.set_ylabel("y (standardized)")
    ax.legend(fontsize=8, ncol=r["n_chains"] + 1)
    fig.tight_layout()
    plt.show()

## 2. Per-chain scalar summary

Permutation-invariant per-chain numbers. A chain that sits at a different value on **any** of these (beyond MC noise) is in a different basin.

| column | meaning |
|---|---|
| `train_ll` | mean over draws of the Gaussian train log-likelihood (per data point) |
| `test_rmse` | posterior-mean-prediction RMSE on held-out data (standardized units) |
| `sigma` | posterior mean of the noise scale |
| `theta_l2` | posterior mean of ‖θ‖ (all weights+biases, excl. log_sigma) |
| `svmax_W1/W2` | posterior mean of the top singular value of each hidden weight matrix |
| `n_active_h1` | posterior mean # of layer-1 units with ‖incoming W‖ above 10% of the max |
| `divs` | divergences in that chain |

In [ ]:
import pandas as pd


def gaussian_ll(pred: np.ndarray, y: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """pred,[S,N]; y,[N]; sigma,[S] -> [S] mean-per-point log-lik."""
    s2 = (sigma ** 2)[:, None]
    ll = -0.5 * np.log(2 * np.pi * s2) - 0.5 * (pred - y[None, :]) ** 2 / s2
    return ll.mean(axis=1)


def chain_scalar_summary(r: dict, sp: dict, n_draws_eval: int = 500) -> pd.DataFrame:
    ls, act = r["layer_sizes"], r["activation"]
    rows = []
    for c in range(r["n_chains"]):
        S = r["samples"][c]
        sel = np.linspace(0, S.shape[0] - 1, min(n_draws_eval, S.shape[0])).round().astype(int)
        draws = S[sel]                                   # [s, D]
        Ws, bs, log_sigma = split_params(draws, ls)
        sigma = np.exp(log_sigma)                        # [s]
        theta_l2 = np.sqrt((draws[:, :-1] ** 2).sum(axis=1))

        row = {"chain": c}
        if sp is not None:
            ptr = forward_batch(sp["X_train"], Ws, bs, act)          # [s, Ntr]
            pte = forward_batch(sp["X_test"], Ws, bs, act)           # [s, Nte]
            row["train_ll"] = gaussian_ll(ptr, sp["y_train"], sigma).mean()
            row["test_rmse"] = np.sqrt(((pte.mean(0) - sp["y_test"]) ** 2).mean())
        row["sigma"] = sigma.mean()
        row["theta_l2"] = theta_l2.mean()
        # top singular value of each hidden weight matrix, averaged over draws
        for li in range(len(Ws) - 1):  # skip the 1xN output layer
            sv = np.linalg.svd(Ws[li], compute_uv=False)             # [s, min(n_out,n_in)]
            row[f"svmax_W{li + 1}"] = sv[:, 0].mean()
        # active layer-1 units: incoming-weight norm > 10% of that draw's max
        w1n = np.linalg.norm(Ws[0], axis=2)                          # [s, n_hidden1]
        active = (w1n > 0.1 * w1n.max(axis=1, keepdims=True)).sum(axis=1)
        row["n_active_h1"] = active.mean()
        row["divs"] = int(r["diverging"][c].sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("chain")


for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    print(f"\n=== {tag} ===")
    display(chain_scalar_summary(r, sp).round(4))

## 3. R-hat / ESS on the predictions (the quantitative answer)

Treat the network output at a set of fixed inputs as the "parameters" and run split-R-hat / ESS across chains.

- **R-hat ≈ 1.0–1.01 across the prediction vector** → the chains agree on the function. Same posterior. (Regardless of what the weights do.)
- **R-hat markedly > 1.1 on many prediction points** → the chains do not agree on the function → different basins.

Also shown: R-hat on `log_sigma` alone (the one identifiable scalar) and, for contrast, R-hat on the raw weights (expected to be terrible — that is the symmetry, not non-convergence).

In [ ]:
def split_rhat(x: np.ndarray) -> np.ndarray:
    """x: [C, N, P] -> [P] split-Rhat. Plain Gelman split-Rhat, no rank-normalisation."""
    C, N, P = x.shape
    half = N // 2
    s = np.concatenate([x[:, :half], x[:, half:2 * half]], axis=0)   # [2C, half, P]
    m, n = s.shape[0], s.shape[1]
    cm = s.mean(axis=1)                                             # [2C, P]
    cv = s.var(axis=1, ddof=1)                                      # [2C, P]
    B = n * cm.var(axis=0, ddof=1)
    W = cv.mean(axis=0)
    var_hat = (n - 1) / n * W + B / n
    return np.sqrt(np.where(W > 0, var_hat / W, np.nan))


def try_arviz_rhat_ess(x: np.ndarray):
    try:
        import arviz as az
        idata = az.convert_to_dataset(x)          # (chain, draw, P)
        return np.asarray(az.rhat(idata).x.values), np.asarray(az.ess(idata).x.values)
    except Exception as e:
        print(f"  (arviz unavailable: {e}); split-Rhat only, ESS=nan")
        return split_rhat(x), np.full(x.shape[-1], np.nan)


rows = []
for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    S = r["samples"]                                # [C, N, D]

    # -- predictions R-hat --
    if sp is not None:
        Xeval = np.concatenate([sp["X_train"], sp["X_test"]], axis=0)
        # subsample draws for speed; keep chain/draw axes
        sel = np.linspace(0, S.shape[1] - 1, 600).round().astype(int)
        pred = np.stack([
            forward_batch(Xeval, *split_params(S[c, sel], r["layer_sizes"])[:2], r["activation"])
            for c in range(r["n_chains"])
        ])                                          # [C, s, Npts]
        pr_rhat, pr_ess = try_arviz_rhat_ess(pred)
        pred_rhat_med = np.nanmedian(pr_rhat)
        pred_rhat_p95 = np.nanpercentile(pr_rhat, 95)
        pred_frac_bad = np.mean(pr_rhat > 1.1)
        pred_ess_med = np.nanmedian(pr_ess)
    else:
        pred_rhat_med = pred_rhat_p95 = pred_frac_bad = pred_ess_med = np.nan

    # -- log_sigma R-hat --
    ls_rhat, ls_ess = try_arviz_rhat_ess(S[:, :, -1:][..., :])
    ls_rhat = float(np.ravel(ls_rhat)[0]); ls_ess = float(np.ravel(ls_ess)[0])

    # -- raw weights R-hat (for contrast) --
    w_rhat = split_rhat(S[:, :, :-1])
    w_rhat = w_rhat[np.isfinite(w_rhat)]

    rows.append({
        "run": tag,
        "pred_rhat_median": pred_rhat_med,
        "pred_rhat_p95": pred_rhat_p95,
        "pred_frac_rhat>1.1": pred_frac_bad,
        "pred_ess_median": pred_ess_med,
        "log_sigma_rhat": ls_rhat,
        "log_sigma_ess": ls_ess,
        "weight_rhat_median": np.nanmedian(w_rhat),
        "weight_rhat_max": np.nanmax(w_rhat),
    })
pd.DataFrame(rows).set_index("run").round(3)

## 4. Sorted per-unit weight-norm profile

For each hidden layer, `‖incoming weights‖` per unit, sorted descending, posterior-mean per chain. This curve is permutation-invariant (sorting removes the labelling).

- **The 5 curves coincide** → every chain uses the same set of unit "strengths" → same basin.
- **Curves differ in shape / number of large-norm units** → chains landed on structurally different networks → different basins (e.g. one uses 15 strong units, another spreads over 30).

In [ ]:
for tag, r in runs.items():
    ls = r["layer_sizes"]
    n_hidden = len(ls) - 2
    fig, axes = plt.subplots(1, n_hidden, figsize=(5.2 * n_hidden, 4), squeeze=False)
    for hidx in range(n_hidden):
        ax = axes[0, hidx]
        for c in range(r["n_chains"]):
            S = r["samples"][c]
            sel = np.linspace(0, S.shape[0] - 1, 400).round().astype(int)
            Ws, _, _ = split_params(S[sel], ls)
            wn = np.linalg.norm(Ws[hidx], axis=2)              # [s, width]
            profile = np.sort(wn, axis=1)[:, ::-1].mean(axis=0)  # sort per draw, then average
            ax.plot(profile, color=CHAIN_COLORS[c % len(CHAIN_COLORS)], lw=1.4, label=f"chain {c}")
        ax.set_title(f"layer {hidx + 1} incoming-weight norm per unit (sorted, post. mean)")
        ax.set_xlabel("unit rank"); ax.set_ylabel("‖W row‖")
        ax.legend(fontsize=8)
    fig.suptitle(tag, y=1.02)
    fig.tight_layout()
    plt.show()

## 5. Within-chain traces of functional scalars

`train_ll`, `log_sigma`, `‖θ‖` per draw, one colour per chain.

- **Flat within each chain, different levels between chains** → each chain locked into its own basin at warmup and never left. The basin question is then purely a between-chain comparison (sections 1–4).
- **A chain that steps between plateaus** → that single chain explored more than one basin (rare for NUTS on a BNN, but this is where you'd see it).
- **All chains flat at the same level** → one basin, everyone agrees.

In [ ]:
def per_draw_scalars(r: dict, sp: dict, thin: int = 5):
    """Returns dict name -> [C, n_thinned] arrays."""
    ls, act = r["layer_sizes"], r["activation"]
    C, N, D = r["samples"].shape
    idx = np.arange(0, N, thin)
    out = {"train_ll": [], "log_sigma": [], "theta_l2": []}
    for c in range(C):
        draws = r["samples"][c, idx]                       # [t, D]
        Ws, bs, log_sigma = split_params(draws, ls)
        out["log_sigma"].append(log_sigma)
        out["theta_l2"].append(np.sqrt((draws[:, :-1] ** 2).sum(axis=1)))
        if sp is not None:
            ptr = forward_batch(sp["X_train"], Ws, bs, act)
            out["train_ll"].append(gaussian_ll(ptr, sp["y_train"], np.exp(log_sigma)))
        else:
            out["train_ll"].append(np.full(len(idx), np.nan))
    return {k: np.stack(v) for k, v in out.items()}, idx


for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    sc, idx = per_draw_scalars(r, sp, thin=5)
    names = ["train_ll", "log_sigma", "theta_l2"]
    fig, axes = plt.subplots(len(names), 1, figsize=(11, 2.1 * len(names)), sharex=True)
    for ax, nm in zip(axes, names):
        for c in range(r["n_chains"]):
            ax.plot(idx, sc[nm][c], color=CHAIN_COLORS[c % len(CHAIN_COLORS)], lw=0.7, alpha=0.85)
        ax.set_ylabel(nm, fontsize=9)
    axes[0].set_title(f"{tag}: within-chain traces of functional scalars", fontsize=11)
    axes[-1].set_xlabel("draw")
    fig.tight_layout()
    plt.show()

## 6. Triangle plots — on permutation-invariant scalars only

Per draw we compute a small vector of quantities that **do not** change under hidden-unit relabelling:

| coord | definition |
|---|---|
| `log_sigma` | noise scale |
| `||theta||` | L2 norm of all weights + biases |
| `svmax_W1` | top singular value of layer-1 weight matrix |
| `svmax_W2` | top singular value of layer-2 weight matrix |
| `train_ll` | mean train log-likelihood of that draw |

Two views:
- **Pooled** (`better_pairs`) — HDR contours. One tight blob = one basin. Several separated blobs = several basins.
- **Per-chain overlay** — each chain a colour. If the pooled plot shows two blobs and each blob is a different colour, the chains split; if every colour covers the same blob, they agree.

In [ ]:
INVARIANT_DRAWS = 1500   # draws per chain fed into the triangle (thinned)


def invariant_coords(r: dict, sp: dict, n_draws: int = INVARIANT_DRAWS):
    """Returns (X [C, s, P], labels). P permutation-invariant scalars per draw."""
    ls, act = r["layer_sizes"], r["activation"]
    C, N, D = r["samples"].shape
    sel = np.linspace(0, N - 1, min(n_draws, N)).round().astype(int)
    n_hidden_mats = len(ls) - 2
    labels = ["log_sigma", "||theta||"] + [f"svmax_W{k+1}" for k in range(n_hidden_mats)]
    if sp is not None:
        labels.append("train_ll")
    per_chain = []
    for c in range(C):
        draws = r["samples"][c, sel]
        Ws, bs, log_sigma = split_params(draws, ls)
        cols = [log_sigma, np.sqrt((draws[:, :-1] ** 2).sum(axis=1))]
        for k in range(n_hidden_mats):
            cols.append(np.linalg.svd(Ws[k], compute_uv=False)[:, 0])
        if sp is not None:
            ptr = forward_batch(sp["X_train"], Ws, bs, act)
            cols.append(gaussian_ll(ptr, sp["y_train"], np.exp(log_sigma)))
        per_chain.append(np.stack(cols, axis=1))       # [s, P]
    return np.stack(per_chain), labels                 # [C, s, P]


def per_chain_overlay_triangle(X, labels, colors, title, max_points=2500):
    """X: [C, s, P]. Lower-tri scatter per chain, diagonal per-chain hist."""
    C, s, P = X.shape
    fig, axes = plt.subplots(P, P, figsize=(3.0 * P, 3.0 * P))
    axes = np.atleast_2d(axes)
    for a in range(P):
        for b in range(P):
            ax = axes[a, b]
            if a == b:
                for c in range(C):
                    ax.hist(X[c, :, a], bins=45, histtype="step", density=True,
                            color=colors[c % len(colors)], lw=1.3)
                ax.set_yticks([]); ax.set_xlabel(labels[a], fontsize=8)
            elif a > b:
                for c in range(C):
                    xs, ys = X[c, :, b], X[c, :, a]
                    if len(xs) > max_points:
                        k = np.linspace(0, len(xs) - 1, max_points).round().astype(int)
                        xs, ys = xs[k], ys[k]
                    ax.scatter(xs, ys, s=4, alpha=0.3, color=colors[c % len(colors)], edgecolors="none")
                if a == P - 1:
                    ax.set_xlabel(labels[b], fontsize=8)
                if b == 0:
                    ax.set_ylabel(labels[a], fontsize=8)
            else:
                ax.set_visible(False)
    handles = [plt.Line2D([0], [0], color=colors[c % len(colors)], lw=2, label=f"chain {c}") for c in range(C)]
    fig.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.9)
    fig.suptitle(title, fontsize=12, y=1.005)
    fig.tight_layout()
    return fig


for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    X, labels = invariant_coords(r, sp)
    print(f"\n=== {tag} === invariant coords: {labels}")

    pooled = X.reshape(-1, X.shape[-1])
    fig, _ = better_pairs(pooled, resol=0.7, labels=labels,
                          title=f"{tag}: pooled invariant scalars (all {r['n_chains']} chains)")
    plt.show()

    fig = per_chain_overlay_triangle(X, labels, CHAIN_COLORS,
                                     title=f"{tag}: invariant scalars, per-chain overlay")
    plt.show()

## 7. Pairwise functional distance between chains

For a fixed input batch, `d(c, c') = RMS over inputs of |mean_pred_c - mean_pred_c'|` (standardized units). A block-diagonal structure = clusters of chains sharing a basin. All-small = one basin.

In [ ]:
for tag, r in runs.items():
    sp = get_split(r["dataset"], r["split_id"])
    if sp is None:
        continue
    Xeval = np.concatenate([sp["X_train"], sp["X_test"]], axis=0)
    mean_preds = []
    for c in range(r["n_chains"]):
        p = chain_predictions(r["samples"][c], Xeval, r["layer_sizes"], r["activation"], 400)
        mean_preds.append(p.mean(0))
    mp = np.stack(mean_preds)                           # [C, Npts]
    C = r["n_chains"]
    Dm = np.zeros((C, C))
    for i in range(C):
        for j in range(C):
            Dm[i, j] = np.sqrt(((mp[i] - mp[j]) ** 2).mean())
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    im = ax.imshow(Dm, cmap="magma")
    for i in range(C):
        for j in range(C):
            ax.text(j, i, f"{Dm[i,j]:.3f}", ha="center", va="center",
                    color="white" if Dm[i, j] < Dm.max() * 0.6 else "black", fontsize=8)
    ax.set_xticks(range(C)); ax.set_yticks(range(C))
    ax.set_xlabel("chain"); ax.set_ylabel("chain")
    ax.set_title(f"{tag}: pairwise functional distance\n(RMS of mean-prediction difference)")
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()

## 8. (optional) Warmup trajectories

Needs `--collect-warmup`. Each chain's functional scalars from prior-init through adaptation and into sampling — shows whether a chain that ends up different branched early or late.

In [ ]:
for tag, r in runs.items():
    if r["warmup_samples"] is None:
        print(f"{tag}: no warmup_samples (run mcmc_test.py with --collect-warmup)")
        continue
    sp = get_split(r["dataset"], r["split_id"])
    ls, act = r["layer_sizes"], r["activation"]
    W, S = r["warmup_samples"], r["samples"]
    nW = r["n_warmup"]
    thin = 5
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for c in range(r["n_chains"]):
        full = np.concatenate([W[c], S[c]], axis=0)[::thin]
        idx = np.arange(0, W.shape[1] + S.shape[1], thin)
        Ws, bs, log_sigma = split_params(full, ls)
        col = CHAIN_COLORS[c % len(CHAIN_COLORS)]
        axes[0].plot(idx, log_sigma, color=col, lw=0.7, alpha=0.85)
        if sp is not None:
            ll = gaussian_ll(forward_batch(sp["X_train"], Ws, bs, act), sp["y_train"], np.exp(log_sigma))
            axes[1].plot(idx, ll, color=col, lw=0.7, alpha=0.85)
    for ax in axes:
        ax.axvline(nW, color="k", ls=":", lw=1)
    axes[0].set_ylabel("log_sigma"); axes[1].set_ylabel("train_ll")
    axes[0].set_title(f"{tag}: warmup + sampling (dotted = end of warmup)", fontsize=11)
    axes[-1].set_xlabel("draw")
    fig.tight_layout()
    plt.show()